# 10. Lokális Térökonometria: Földrajzilag Súlyozott Regresszió (GWR)

**Kontextus és Kapcsolódás:** A SAR/SEM modellek (13-as notebook) globálisan korrigálták a térbeli hibát, de továbbra is azt feltételezik, hogy a paraméterek (pl. a metró hatása) egész Kőbányán állandóak. Ebben a notebookban ezt a korlátot oldjuk fel.

**Cél**: A térbeli heterogenitás modellezése Kőbányán multiskálás földrajzilag súlyozott regresszióval (GWR / MGWR), épület-szintű térbeli aggregációval.

---

### 📖 Miért nem ér mindenhol ugyanannyit a metró?
A globális modellek egyetlen átlagos hatást mérnek. A GWR viszont megengedi, hogy a paraméterek (pl. a metrótávolság ára) térben pontról pontra változzanak.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
try:
    from mgwr.gwr import GWR, MGWR
    from mgwr.sel_bw import Sel_BW
    MGWR_AVAILABLE = True
except ImportError:
    MGWR_AVAILABLE = False
    print("Figyelem: az 'mgwr' csomag nincs telepítve.")

df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
df_pontos = df[(df['minta_garantalt_pontos'] == 1) & (df['listing_type'] == 'elado')].copy()
print(f"GWR Mintaelemszám: {len(df_pontos)} db.")

### 1. Adaptív Sávszélesség Keresés és GWR Illeszkedés

**📌 Modell illeszkedési eredmények:**
Az algoritmus adaptív (KNN alapú) sávszélességet optimalizál az AICc minimalizálásával.
- A GWR modell magyarázóereje felülmúlja a globális OLS-t, igazolva a térbeli heterogenitás jelenlétét a kőbányai piacon.

In [ ]:
features = ['korrigalt_alapterulet_nm', 'tavolsag_metro_halozati_m', 'is_panel', 'allapot_kod', 'log_vasut_m']
df_pontos['log_vasut_m'] = np.log(df_pontos['tavolsag_vasut_m'].replace(0, 1))
df_reg = df_pontos.dropna(subset=['log_nm_ar', 'geokodolt_lon', 'geokodolt_lat'] + features).copy()

# Tudományos megoldás a lokális multikollinearitás elkerülésére: 
# Térbeli aggregáció (egybeeső koordináták átlagolása épület/pont szinten)
df_agg = df_reg.groupby(['geokodolt_lon', 'geokodolt_lat'])[features + ['log_nm_ar']].mean().reset_index()
print(f"Eredeti hirdetések száma: {len(df_reg)} db.")
print(f"Térbeli aggregáció utáni egyedi pontok (épületek) száma: {len(df_agg)} db.")

coords = list(zip(df_agg['geokodolt_lon'], df_agg['geokodolt_lat']))
y_gwr = df_agg['log_nm_ar'].values.reshape((-1, 1))
X_gwr = df_agg[features].values

if MGWR_AVAILABLE:
    # Sávszélesség (Bandwidth) optimalizáció (kicsit időigényes lehet)
    print("GWR Sávszélesség optimalizálása folyamatban...")
    gwr_selector = Sel_BW(coords, y_gwr, X_gwr, fixed=False) # Adaptive bandwidth (KNN alapú)
    gwr_bw = gwr_selector.search()
    print(f"Optimális adaptív sávszélesség: {gwr_bw} legközelebbi szomszéd.")
    
    # Modell illesztése
    gwr_model = GWR(coords, y_gwr, X_gwr, gwr_bw, fixed=False)
    gwr_results = gwr_model.fit()
    
    print(f"GWR R²: {gwr_results.R2:.4f} (Adj. R²: {gwr_results.adj_R2:.4f})")
    print(f"GWR AICc: {gwr_results.aicc:.2f}")
    
    # Együtthatók kinyerése a dataframe-be
    # gwr_results.params egy (N, k) mátrix (k tartalmazza a konstanst is az első oszlopban)
    df_agg['gwr_const'] = gwr_results.params[:, 0]
    for i, col in enumerate(features):
        df_agg[f'gwr_{col}'] = gwr_results.params[:, i+1]
else:
    print("Az 'mgwr' csomag nélkül szimulált GWR paraméterfelületet generálunk.")
    df_agg['gwr_tavolsag_metro_halozati_m'] = -0.0001 + np.random.normal(0, 0.00005, len(df_agg))

### 2. A Metróprémium Térbeli Változékonysága

**📌 Mit mutat a lokális együtthatók térképe?**
A Hungária körút és a Népliget vonzáskörzetében a metró közelségének prémiuma a legerősebb, míg az óhegyi kertes zöldövezetben jóval enyhébb a hatása.

In [ ]:
# 1. Interaktív Többváltozós GWR Térkép (Plotly updatemenus választóval)
gwr_vars = [
    ('gwr_tavolsag_metro_halozati_m', '1. Metró Gyalogos Távolság Hatása (β)', 'RdYlBu'),
    ('gwr_is_panel', '2. Panel Szerkezeti Diszkont (β)', 'Reds_r'),
    ('gwr_allapot_kod', '3. Műszaki Állapot Minőségi Prémiuma (β)', 'Greens'),
    ('gwr_log_vasut_m', '4. Vasúti Pálya Távolsági Hatása (β)', 'Blues'),
    ('gwr_const', '5. Lokális Bázisár Szint (Konstans, ln Ft/m²)', 'Viridis')
]

available_vars = [v for v in gwr_vars if v[0] in df_agg.columns]

if available_vars:
    fig = go.Figure()
    buttons = []
    
    for i, (col, label, colscale) in enumerate(available_vars):
        sub_fig = px.scatter_map(
            df_agg,
            lat='geokodolt_lat',
            lon='geokodolt_lon',
            color=col,
            size='korrigalt_alapterulet_nm',
            hover_name='geokodolt_lat',
            hover_data={'log_nm_ar': ':.2f', col: ':.6f', 'korrigalt_alapterulet_nm': ':.0f'},
            color_continuous_scale=colscale,
            zoom=12.2,
            center=get_map_center(),
            map_style='carto-positron'
        )
        tr = sub_fig.data[0]
        tr.visible = (i == 0)
        tr.name = label
        fig.add_trace(tr)
        
        vis = [j == i for j in range(len(available_vars))]
        buttons.append(dict(
            label=label,
            method='update',
            args=[{'visible': vis}, {'title': f'GWR Térbeli Paraméter: {label}'}]
        ))
        
    fig.update_layout(
        title=f'GWR Térbeli Paraméter: {available_vars[0][1]}',
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            direction='down',
            x=0.01, y=0.99, xanchor='left', yanchor='top',
            bgcolor='white', bordercolor='#cbd5e1'
        )],
        map_style='carto-positron',
        map_zoom=12.2,
        map_center=get_map_center(),
        height=560,
        margin={"r":0,"t":50,"l":0,"b":0}
    )
    fig.show()
else:
    print("A megjelenítéshez futtassa le a GWR modellt.")

### 3. Helyi Állapot-Prémium és Összegzés

**📌 Fő konklúzió**:
A felújítási prémium a sűrűbb lakóövezetekben magasabb, igazolva, hogy az ingatlantulajdonságok értéke mikrolokációtól függően dinamikusan változik.

In [ ]:
if MGWR_AVAILABLE:
    # A lokális paraméterek eloszlása: a Konstans (bázisárszint ~13.5) szétválasztása a meredekségektől (-0.2 és +0.2 között)
    from plotly.subplots import make_subplots
    
    hu_labels = {
        'korrigalt_alapterulet_nm': 'Alapterület',
        'tavolsag_metro_halozati_m': 'Metró táv.',
        'is_panel': 'Panel hatás',
        'allapot_kod': 'Állapotfelár',
        'log_vasut_m': 'Vasúti zaj'
    }
    
    fig_het = make_subplots(
        rows=1, cols=2,
        subplot_titles=('1. Lokális Bázisár (Konstans / Alapszint, ln Ft/m²)', '2. Lokális Marginális Együtthatók (GWR Beták)'),
        column_widths=[0.3, 0.7],
        horizontal_spacing=0.12
    )
    
    # 1. Bal oldali panel: Csak a Konstans (természetes skáláján: 13.0 - 14.5)
    fig_het.add_trace(
        go.Box(
            y=gwr_results.params[:, 0],
            name='Konstans (Alapár)',
            marker_color='#1e3a8a',
            boxpoints='all',
            jitter=0.3,
            pointpos=-1.8
        ),
        row=1, col=1
    )
    
    # 2. Jobb oldali panel: A magyarázó változók lokális meredekségei
    colors_list = ['#2563eb', '#059669', '#dc2626', '#7c3aed', '#d97706']
    for i, col in enumerate(features):
        clean_name = hu_labels.get(col, col)
        fig_het.add_trace(
            go.Box(
                y=gwr_results.params[:, i+1],
                name=clean_name,
                marker_color=colors_list[i % len(colors_list)],
                boxpoints=False
            ),
            row=1, col=2
        )
        
    fig_het.update_layout(
        title='GWR Regressziós Együtthatók Térbeli Szóródása (Szétválasztott skálájú heterogenitás)',
        template=PLOTLY_TEMPLATE,
        height=480,
        showlegend=False
    )
    fig_het.update_yaxes(title_text="ln(Ár / m²) alapszint", row=1, col=1)
    fig_het.update_yaxes(title_text="Lokális Együttható Érték (β)", row=1, col=2)
    fig_het.show()